In [29]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import random

from data import load_coco_data, CocoDataset, decode_captions
from RNN import CaptioningRNN, CaptioningRNNTrainer

In [31]:
random.seed(0)
torch.manual_seed(0)

BATCH_SIZE=32
NUM_WORKERS=1

model_save_root = "../../Models/RNN"
os.makedirs(model_save_root,exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

In [32]:
# COCO captioning

data_root = '../../Data/COCO_captioning'

zip_path = os.path.join(data_root, "coco_captioning.zip")

if os.path.isfile(zip_path):
    print("COCO data exists!")
else:
    print("downloading COCO dataset")
    !wget https://cs231n.stanford.edu/coco_captioning.zip -P {data_root}
    !unzip -d {data_root} {zip_path}

COCO data exists!


In [33]:
data = load_coco_data(os.path.join(data_root,'coco_captioning'), pca_features=True)
word_to_idx = data['word_to_idx']
feature_dim = data['train_features'].shape[1]

train_dataset = CocoDataset(data, split='train')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

# RNN

In [ ]:
model = CaptioningRNN(
    word_to_idx=word_to_idx,
    img_feat_dim=feature_dim,
    wordvec_dim=128,
    h_dim=128,
    dtype=torch.float32
)

model = model.to(DEVICE) 

optimizer = optim.Adam(model.params.values(), lr=1e-3, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

trainer = CaptioningRNNTrainer(model=model, 
                               optimizer=optimizer,
                               scheduler=scheduler,
                               device=DEVICE)

trainer.train(
    train_dataloader=train_loader,
    num_epochs=30,
    log_interval=50,
    save_path=os.path.join(model_save_root,'CaptionRNN.pth')
)

In [35]:
model = CaptioningRNN(
    word_to_idx=word_to_idx,
    img_feat_dim=feature_dim,
    wordvec_dim=128,
    h_dim=128,
    dtype=torch.float32
)

params = torch.load(os.path.join(model_save_root,'CaptionRNN.pth'), map_location='cpu') 

model.params = params
model.embedding.W = params['W_embed']
model.projection.W = params['W_proj']
model.projection.b = params['b_proj']
model.rnn_cell.Wx = params['Wx']
model.rnn_cell.Wh = params['Wh']
model.rnn_cell.b = params['b']
model.vocabulary.W = params['W_vocab']
model.vocabulary.b = params['b_vocab']

model.to(DEVICE)

val_features = data['val_features'][:4]         # (4, img_feat_dim)
val_captions_true = data['val_captions'][:4]    # (4, T) GT

features_tensor = torch.tensor(val_features, dtype=torch.float32).to(DEVICE)

pred_captions = model.sample(features_tensor)   # (4, max_length)

idx_to_word = data['idx_to_word']   
for i in range(4):
    pred = decode_captions(pred_captions[i:i+1], idx_to_word)[0]
    true = decode_captions(val_captions_true[i:i+1], idx_to_word)[0]
    print(f"Image {i} prediction: {pred}")
    print(f"      GT: {true}\n")

Image 0 prediction: a man is <UNK> a <UNK> <UNK> in a <UNK> <END>
      GT: <START> a bicycle <UNK> with a clock as the front <UNK> <END>

Image 1 prediction: a man is <UNK> a <UNK> <UNK> in a <UNK> <END>
      GT: <START> a black <UNK> motorcycle parked in front of a <UNK> <END>

Image 2 prediction: a man is <UNK> a <UNK> <UNK> in a <UNK> <END>
      GT: <START> a room with blue walls and a white sink and door <END>

Image 3 prediction: a man is <UNK> a <UNK> <UNK> in a <UNK> <END>
      GT: <START> a car that <UNK> to be parked <UNK> behind a <UNK> parked car <END>



/tmp/ipykernel_3411719/2913873187.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params = torch.load(os.path.join(model_save_root,'CaptionRNN.pth'), map_location='cpu')